<a href="https://colab.research.google.com/github/rgonzalezp-ucsc/Ciencia-de-Datos_UCSC_Rodrigo_G/blob/main/S2_Notebook_Laboratorio_CDD_Gonzalez_Rodrigo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | _RODRIGO I. GONZALEZ P_ |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [1]:
import pandas as pd
import numpy as np

RUTA = '/content/'

SEMILLA = 20262
np.random.seed(SEMILLA)

ARCHIVO = 'day.csv'
SEPARADOR = ','

df = pd.read_csv(RUTA + ARCHIVO, sep=SEPARADOR)

print('Forma:', df.shape)
display(df.head())

Forma: (731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [2]:
df.dtypes

,0
instant,int64
dteday,object
season,int64
yr,int64
mnth,int64
holiday,int64
weekday,int64
workingday,int64
weathersit,int64
temp,float64


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [3]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
instant,0,0.0
dteday,0,0.0
season,0,0.0
yr,0,0.0
mnth,0,0.0
holiday,0,0.0
weekday,0,0.0
workingday,0,0.0
weathersit,0,0.0
temp,0,0.0


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [4]:
# Actividad 1.1: investigar la distribución de los faltantes

columna_max = faltantes["faltantes"].idxmax()
cantidad_max = int(faltantes["faltantes"].max())

print("Columna con más faltantes:", columna_max)
print("Cantidad máxima de faltantes:", cantidad_max)

if cantidad_max == 0:
    print(
        "\nConclusión: no se detectaron valores faltantes "
        "en ninguna columna. No corresponde analizar su "
        "distribución por grupos ni realizar imputaciones "
        "por faltantes detectados mediante isna()."
    )
else:
    grupo = "season"

    tabla = pd.crosstab(
        df[grupo],
        df[columna_max].isna(),
        margins=True
    )

    display(tabla)


Columna con más faltantes: instant
Cantidad máxima de faltantes: 0

Conclusión: no se detectaron valores faltantes en ninguna columna. No corresponde analizar su distribución por grupos ni realizar imputaciones por faltantes detectados mediante isna().


### 1.2 Exactitud — rangos y valores imposibles

In [5]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
instant,731.0,366.00,211.17,1.00,183.50,366.00,548.50,731.00
season,731.0,2.50,1.11,1.00,2.00,3.00,3.00,4.00
yr,731.0,0.50,0.50,0.00,0.00,1.00,1.00,1.00
mnth,731.0,6.52,3.45,1.00,4.00,7.00,10.00,12.00
holiday,731.0,0.03,0.17,0.00,0.00,0.00,0.00,1.00
weekday,731.0,3.00,2.00,0.00,1.00,3.00,5.00,6.00
workingday,731.0,0.68,0.47,0.00,0.00,1.00,1.00,1.00
weathersit,731.0,1.40,0.54,1.00,1.00,1.00,2.00,3.00
temp,731.0,0.50,0.18,0.06,0.34,0.50,0.66,0.86
atemp,731.0,0.47,0.16,0.08,0.34,0.49,0.61,0.84


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [6]:
# Actividad 1.2: búsqueda de valores centinela

centinelas = [-1, 0, 999, 9999, -999]

for col in df.select_dtypes(include="number").columns:
    encontrados = {}

    for valor in centinelas:
        cantidad = int((df[col] == valor).sum())

        if cantidad > 0:
            encontrados[valor] = cantidad

    if encontrados:
        print(f"{col}: {encontrados}")

# Revisar posibles centinelas de texto
centinelas_texto = ["S/I", "NULL", "sin dato"]

for col in df.select_dtypes(include="object").columns:
    valores = df[col].astype(str).str.strip().str.upper()

    for valor in centinelas_texto:
        cantidad = int((valores == valor.upper()).sum())

        if cantidad > 0:
            print(f"{col}: '{valor}' aparece {cantidad} veces")


yr: {0: 365}
holiday: {0: 710}
weekday: {0: 105}
workingday: {0: 231}
hum: {0: 1}


### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [7]:
# Regla 1: el total de arriendos debe ser igual
# a la suma de los arriendos ocasionales y registrados.

regla1 = df["cnt"] == (df["casual"] + df["registered"])

print("REGLA 1: cnt = casual + registered")
print("Registros que cumplen:", int(regla1.sum()))
print("Registros que no cumplen:", int((~regla1).sum()))


# Regla 2: el mes y el año deben coincidir con la fecha registrada.

fechas = pd.to_datetime(df["dteday"], errors="coerce")

mes_correcto = fechas.dt.month == df["mnth"]
anio_correcto = fechas.dt.year == (2011 + df["yr"])

regla2 = mes_correcto & anio_correcto

print("\nREGLA 2: coherencia entre fecha, mes y año")
print("Registros que cumplen:", int(regla2.sum()))
print("Registros que no cumplen:", int((~regla2).sum()))


REGLA 1: cnt = casual + registered
Registros que cumplen: 731
Registros que no cumplen: 0

REGLA 2: coherencia entre fecha, mes y año
Registros que cumplen: 731
Registros que no cumplen: 0


### 1.4 Unicidad

In [ ]:
print('Duplicados exactos:', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
# print('ID repetidos:', df['TU_ID'].duplicated().sum())

### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [8]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Construir una clave normalizada a partir de la fecha
clave = pd.to_datetime(
    df["dteday"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# Contar fechas inválidas y duplicadas
print("Fechas inválidas:", int(clave.isna().sum()))
print("Fechas duplicadas:", int(clave.duplicated().sum()))

# Mostrar registros con fechas repetidas
mascara = clave.duplicated(keep=False)

if mascara.any():
    display(
        df.loc[mascara]
        .sort_values("dteday")
    )
else:
    print("No se detectaron fechas duplicadas.")


Fechas inválidas: 0
Fechas duplicadas: 0
No se detectaron fechas duplicadas.


### 1.5 Oportunidad

In [10]:
# Actividad 1.5: evaluar la oportunidad de los datos

fechas = pd.to_datetime(df["dteday"], errors="coerce")

fecha_minima = fechas.min()
fecha_maxima = fechas.max()
fechas_validas = fechas.notna().sum()

print("Primer registro:", fecha_minima)
print("Último registro:", fecha_maxima)
print("Fechas válidas:", int(fechas_validas))
print("Registros totales:", len(df))

if fechas_validas == len(df):
    print("Todas las fechas son válidas.")
else:
    print("Hay fechas que requieren revisión.")

print(
    "\nInterpretación: el conjunto permite estudiar "
    "patrones históricos de arriendos durante el período observado. "
    "No debe asumirse que los resultados representan necesariamente "
    "la demanda actual."
)


Primer registro: 2011-01-01 00:00:00
Último registro: 2012-12-31 00:00:00
Fechas válidas: 731
Registros totales: 731
Todas las fechas son válidas.

Interpretación: el conjunto permite estudiar patrones históricos de arriendos durante el período observado. No debe asumirse que los resultados representan necesariamente la demanda actual.


---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [11]:
# Investigar el registro con humedad igual a cero

casos_hum_cero = df[df["hum"] == 0]

print("Registros con hum = 0:", len(casos_hum_cero))

display(casos_hum_cero)


Registros con hum = 0: 1


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
68,69,2011-03-10,1,0,3,0,4,1,3,0.389091,0.385668,0.0,0.261877,46,577,623


### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [12]:
# Actividad 2.2: revisar las categorías del conjunto

columnas_categoricas = [
    "season",
    "yr",
    "mnth",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

for columna in columnas_categoricas:
    print(f"\n--- {columna} ---")
    print("Valores únicos:", sorted(df[columna].dropna().unique()))
    print("Cantidad de categorías:", df[columna].nunique())



--- season ---
Valores únicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Cantidad de categorías: 4

--- yr ---
Valores únicos: [np.int64(0), np.int64(1)]
Cantidad de categorías: 2

--- mnth ---
Valores únicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Cantidad de categorías: 12

--- holiday ---
Valores únicos: [np.int64(0), np.int64(1)]
Cantidad de categorías: 2

--- weekday ---
Valores únicos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Cantidad de categorías: 7

--- workingday ---
Valores únicos: [np.int64(0), np.int64(1)]
Cantidad de categorías: 2

--- weathersit ---
Valores únicos: [np.int64(1), np.int64(2), np.int64(3)]
Cantidad de categorías: 3


### ✏️ Actividad 2.3 — Duplicados

In [13]:
# Actividad 2.3: detectar duplicados

# 1. Contar filas completamente duplicadas
duplicados_exactos = df.duplicated().sum()

print("Duplicados exactos:", int(duplicados_exactos))

# 2. Comprobar si existen fechas repetidas
duplicados_fecha = df["dteday"].duplicated().sum()

print("Fechas duplicadas:", int(duplicados_fecha))

# 3. Mostrar posibles duplicados por fecha
if duplicados_fecha > 0:
    display(
        df[df["dteday"].duplicated(keep=False)]
        .sort_values("dteday")
    )
else:
    print("No hay fechas repetidas.")


Duplicados exactos: 0
Fechas duplicadas: 0
No hay fechas repetidas.


### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [14]:
# Actividad 2.4: revisar y corregir tipos de datos

print("Tipos de datos originales:")
print(df.dtypes)

# Convertir la fecha a formato datetime
df["dteday"] = pd.to_datetime(
    df["dteday"],
    errors="coerce"
)

# Asegurar que las variables de conteo sean numéricas
columnas_numericas = [
    "instant", "casual", "registered", "cnt"
]

for columna in columnas_numericas:
    df[columna] = pd.to_numeric(
        df[columna],
        errors="coerce"
    )

print("\nTipos de datos después de la revisión:")
print(df.dtypes)

print("\nValores faltantes después de la conversión:")
print(df[["dteday"] + columnas_numericas].isna().sum())


Tipos de datos originales:
instant         int64
dteday         object
season          int64
yr              int64
mnth            int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
casual          int64
registered      int64
cnt             int64
dtype: object

Tipos de datos después de la revisión:
instant                int64
dteday        datetime64[ns]
season                 int64
yr                     int64
mnth                   int64
holiday                int64
weekday                int64
workingday             int64
weathersit             int64
temp                 float64
atemp                float64
hum                  float64
windspeed            float64
casual                 int64
registered             int64
cnt                    int64
dtype: object

Valores faltantes después de la conversión:
dteday        0
instant       0
casual      

### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [15]:
# Actividad 2.5: identificar valores atípicos

# Revisar los extremos de las variables de arriendos
columnas = ["casual", "registered", "cnt"]

print("RESUMEN ESTADÍSTICO DE LOS ARRIENDOS")
display(df[columnas].describe().T.round(2))

# Identificar valores extremos mediante el criterio del rango intercuartílico
for col in columnas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    atipicos = df[
        (df[col] < limite_inferior) |
        (df[col] > limite_superior)
    ]

    print(f"\nVariable: {col}")
    print("Límite inferior:", round(limite_inferior, 2))
    print("Límite superior:", round(limite_superior, 2))
    print("Cantidad de valores atípicos:", len(atipicos))

# Cruzar el total de arriendos con la temperatura
print("\nDÍAS CON MAYOR CANTIDAD DE ARRIENDOS")
display(
    df.nlargest(5, "cnt")[
        ["dteday", "temp", "hum", "windspeed",
         "casual", "registered", "cnt"]
    ]
)


RESUMEN ESTADÍSTICO DE LOS ARRIENDOS


,count,mean,std,min,25%,50%,75%,max
casual,731.0,848.18,686.62,2.0,315.5,713.0,1096.0,3410.0
registered,731.0,3656.17,1560.26,20.0,2497.0,3662.0,4776.5,6946.0
cnt,731.0,4504.35,1937.21,22.0,3152.0,4548.0,5956.0,8714.0



Variable: casual
Límite inferior: -855.25
Límite superior: 2266.75
Cantidad de valores atípicos: 44

Variable: registered
Límite inferior: -922.25
Límite superior: 8195.75
Cantidad de valores atípicos: 0

Variable: cnt
Límite inferior: -1054.0
Límite superior: 10162.0
Cantidad de valores atípicos: 0

DÍAS CON MAYOR CANTIDAD DE ARRIENDOS


,dteday,temp,hum,windspeed,casual,registered,cnt
623,2012-09-15,0.608333,0.501667,0.247521,3160,5554,8714
637,2012-09-29,0.542500,0.542917,0.227604,2589,5966,8555
630,2012-09-22,0.650000,0.646667,0.283583,2512,5883,8395
447,2012-03-23,0.601667,0.694167,0.116300,2469,5893,8362
504,2012-05-19,0.600000,0.456250,0.083975,3410,4884,8294


_Conclusión sobre los atípicos:_El método del rango intercuartílico detectó 44 valores atípicos en la variable `casual`, mientras que no detectó valores atípicos en `registered` ni en `cnt`. Los valores extremos no se consideran automáticamente errores, ya que pueden representar días con una demanda elevada de bicicletas. Por este motivo, se conservarán los registros y no se eliminarán observaciones sin evidencia de errores de registro. Se recomienda analizar estos casos junto con variables como la temperatura, la humedad y la fecha.




---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

In [16]:
# Actividad 3.1: creación de atributos derivados

# Crear una variable con el total calculado de arriendos
df["total_verificado"] = df["casual"] + df["registered"]

# Comprobar si coincide con el total original
coinciden = (df["total_verificado"] == df["cnt"]).all()

print("¿El total calculado coincide con cnt?:", coinciden)

# Mostrar algunos registros
display(
    df[["dteday", "casual", "registered", "cnt", "total_verificado"]].head()
)


¿El total calculado coincide con cnt?: True


,dteday,casual,registered,cnt,total_verificado
0,2011-01-01,331,654,985,985
1,2011-01-02,131,670,801,801
2,2011-01-03,120,1229,1349,1349
3,2011-01-04,108,1454,1562,1562
4,2011-01-05,82,1518,1600,1600


_Por qué esta variable aporta a mi pregunta:_La variable total_verificado permite comprobar la consistencia de los datos, verificando que la suma de los arriendos ocasionales y registrados coincida con el total diario. Esto ayuda a detectar posibles inconsistencias y aumenta la confianza en los datos antes de realizar análisis posteriores._



### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [ ]:
# Tu código aquí. Referencia:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# df_cod = pd.get_dummies(df, columns=['TU_COLUMNA_CATEGORICA'], drop_first=True)


---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [ ]:
def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """Flujo completo de limpieza de mi conjunto. Devuelve (df_limpio, informe)."""
    np.random.seed(semilla)
    d = pd.read_csv(ruta, sep=sep)
    informe = {'filas_iniciales': len(d)}

    # 1 · centinelas
    # 2 · categorías
    # 3 · duplicados
    # 4 · tipos
    # 5 · atípicos
    # 6 · atributos derivados

    informe['filas_finales'] = len(d)
    return d, informe


limpio, informe = limpiar(RUTA + ARCHIVO)
print(informe)

In [ ]:
# Comprobación: ejecuta la función DOS VECES y verifica que da lo mismo.
a, _ = limpiar(RUTA + ARCHIVO)
b, _ = limpiar(RUTA + ARCHIVO)
print('¿Reproducible?', a.equals(b))

In [ ]:
# Guarda el conjunto limpio. Este archivo es el insumo de la semana 3.
limpio.to_csv(RUTA + 'mi_conjunto_limpio.csv', sep=';', index=False)
print('Guardado:', limpio.shape)

---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:** _…_

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_